## Tutorial

# Calculate and Compare Canopy Water Content

## Second of two notebooks

### Authors: Hannah Rieder, Randi Neff, Bridget Hass

## **bolded text below was added by Hannah on 7/31/25**

In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface **bi**directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product.

## The objectives of this tutorial (divided between two notebooks) are to:

* Use co-located data from NEON and EMIT
* **Crop EMIT reflectance data to NEON tile boundaries**
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA The data provided with this tutorial were derived from existing code at:

* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* Land Processes Distributed Active Archive Center (LP DAAC).

Additional data will be downloaded programmatically within this tutorial.

## What we should have after completing notebook 1:

* two EMIT cropped datasets (one for the burned tile and one for the unburned tile) exported to netcdf files **saved to the reflectance directory at `../data/refl/`**
* two NEON reflectance datasets (one for the burned tile and one for the unburned tile). These datasets will already have been converted from hdf5 format into xarray, have the scale factor applied, have bad bands set to NaN, have necessary data types turned from float64 to float32, and be exported to netcdf files **saved to the reflectance directory at `../data/refl/`**

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison

0. Import Standard Packages
1. Setup

   1.1 Create Data and Scripts (modules) Directories

   1.2 Download and Import Necessary Scripts

   1.3 Download and Open the Refractive Index of Liquid Water per Wavelength CSV
3. Open NEON and EMIT Reflectance Data
4. Calculate Canopy Water Content (CWC)
5. Compare CWC Datasets

## Reference/credit to:
The existing [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html) tutorial in the [NASA VITALS Repository](https://github.com/nasa/VITALS). **That existing notebook is a tutorial for how to calculate Equivalent Water Thickness (EWT) or Canopy Water Content (CWC) from EMIT L2A reflectance data over a nature preserve near Santa Barbara, California. We followed that tutorial with EMIT L2A reflectance data over the the SOAP field site while exploring CWC and used it to help create this notebook. Specifically, section "3.5 CWC Calculation of an ROI" of the NASA VITALS existing tutorial and the ewt_calc.py script were used to calculate CWC below.**

### 0. Import Standard Packages

In [ ]:
# Import Packages
import os, sys # Python module to create and acces file paths
import pathlib
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np # Work with multi-dimensional arrays
import xarray as xr # Work with labelled multi-dimenstional arrays
from osgeo import gdal # Work with raster and vector geospatial data
import rasterio as rio # Work with geospatial raster data
import rioxarray as rxr # Work with raster arrays
from matplotlib import pyplot as plt # Plotting data
import hvplot.xarray # Plot multi-dimensional arrays
import hvplot.pandas # Plot DataFrames/Series
import pandas as pd # Work with DataFrames
import geopandas as gpd # Work with geospatial shapefiles
import earthaccess # Search for, download, & stream NASA earth data
from tqdm.notebook import tqdm # Progress bars on loops
import requests
from scipy.optimize import least_squares # Nonlinear least-squares
import holoviews as hv

#import neonutilities as nu # Work with NEON reflectance data
import h5py # Work with NEON reflectance data

### 1. Setup

In the setup section, we will do three things:
1. create directories to store the input and output data and scripts for this project,
2. download and import the extra necessary scripts needed for this tutorial, and
3. download a comma separated value (CSV) file needed for the CWC calculation function.

### 1.1 Create Data and Scripts (modules) Directories

#### *If you completed Tutorial Notebook 01, you will already have the data and scripts directories created. It is recommended to still run the code in 1.1 to verify that the directories have been made.*

The directories we will make are: an overarching data directory, a reflectance data directory, a CWC data directory, a shapefiles data directory, and a modules directory.

The overarching data directory will contain the reflectance and CWC data directories and a CSV file containing lab measurements of the complex refractive index of liquid water. 

The reflectance data directory should contain the two EMIT cropped datasets (one for the burned tile and one for the unburned tile) and the two NEON reflectance datasets (one for the burned tile and one for the unburned tile) created in Tutorial Notebook 01. All of these datasets should be NetCDF files.

The shapefiles data directory will contain shapefiles for the SOAP site and the tiles.

The CWC data directory will be where we store the results of this tutorial notebook: the CWC calculations of the burned and unburned tiles calculated using the cropped EMIT and NEON reflectance data.

The modules directory must be in the same folder as where you have these tutorial notebooks stored for some of the imported functions to work. In this modules directory, we will manually download some Python scripts (.py files). The scripts contain various functions we'll use in this tutorial.

The CWC calculation functions (calc_ewt and calc_ewt_neon) are expecting the data and the k_liquid_water_ice.csv file to be stored in a directory that is at the same file level as where you have this notebook stored. In the cell below, the `data_dir = r"../data"` code ensures that the data directory will be at the same file level as where this tutorial notebook is stored.

Here is a visual of how the directory and file structure will look once these directories are created and files are downloaded:

```
project-root/
│
├── data/                     # Main folder for data
│   ├── cwc/                  # Subfolder for canopy water content (CWC) data genearted in tutorial_notebook_02
│   │   ├── emit_burn_cwc.nc               
│   │   ├── emit_burn_cwc.tif               
│   │   ├── emit_unburn_cwc.nc
│   │   ├── emit_unburn_cwc.tif                           
│   │   ├── neon_burn_cwc.nc               
│   │   ├── neon_burn_cwc.tif               
│   │   ├── neon_unburn_cwc.nc               
│   │   └── neon_unburn_cwc.tif          
│   │
│   ├── refl/                 # Subfolder for reflectance data generated in tutorial_notebook_01
│   │   ├── emit_burn_refl.nc
│   │   ├── emit_unburn_refl.nc
│   │   ├── neon_burn_refl_float32.nc
|   |   └── neon_unburn_refl_float32.nc
|   ├── shapefiles/
|   |   ├── AOPflightboxes
|   └── k_liquid_water_ice.csv
│
└── notebooks/                # Subfolder for modules and tutorial notebooks
    ├── modules/              # Subfolder for Python scripts for processing and analysis
    │   ├── .ipynb_checkpoints
    |   ├── __pycache__
    |   ├── __init__
    |   ├── ewt_tools.py
    |   ├── ewt_calc2.py
    |   └── test_functions.py
    ├── tutorial_notebook_01.ipynb
    └── tutorial_notebook_02.ipynb
```

In [ ]:
# Define the file path for the data directory
data_dir = r"../data"

# Create/check for the data_dir
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f'data directory made here: {data_dir}')
else:
    print(f'data directory already exists here: {data_dir}')

In [ ]:
# List of directories names to make
dir_list = ["refl", "cwc"]

# Define the root path where the directories will be created
root_path = data_dir

# Use a for loop to create/check for the directories in the dir_list
for dir_name in dir_list:
    full_path = os.path.join(root_path, dir_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f'directory made here: {full_path}')
    else:
        print(f'directory already exists here: {full_path}')

In [ ]:
# Define the file path for the modules directory
modules_dir = r"./modules"

# Create/check for the modules_dir
if not os.path.exists(modules_dir):
    os.makedirs(modules_dir)
    print(f'modules directory made here: {modules_dir}')
else:
    print(f'modules directory already exists here: {modules_dir}')

### 1.2 Download and Import Necessary Scripts

#### *If you completed Tutorial Notebook 01, you will already have the ewt_calc2.py, test_functions.py, and emit_tools.py scripts downloaded and can skip the download instructions below. However, you will still need to run the code in the cell below to import necessary functions from those scripts.*

The Python scripts that we will put in the modules directory (ewt_calc2.py, emit_tools.py, and test_functions.py) contain functions that will allow us to calculate and visualize CWC and see a directory's contents. The emit_tools.py script is required for the functions in the ewt_calc2.py script to work. We can access and download the scripts with the following steps:

1. Follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/ewt_calc2.py) to access the ewt_calc2.py script. **MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**
2. Manually download the raw file to your computer.
3. Move the ewt_calc2.py script into the modules directory we made above (`modules_dir = r"./modules"`). **It is important that the ewt_calc2.py script is in the modules_dir; the import code below expects the script to be in the modules_dir.**
4. Run the code in the cell below to import functions from the ewt_calc2.py script into this notebook.
5. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/test_functions.py) to access the test_functions.py script.**MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**
6. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/emit_tools.py) to access the emit_tools.py script.**MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**

In [ ]:
# Import functions from the python scripts in the modules directory
from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
from modules.emit_tools import emit_xarray # Open EMIT datasets as xarray.Dataset
from modules.ewt_calc2 import calc_ewt, calc_ewt_neon # Canopy water content fxn

### 1.3 Download and Open the Refractive Index of Liquid Water per Wavelength CSV

The CWC calculation functions (calc_ewt and calc_ewt_neon) requires this CSV file to work. The name of the CSV file is k_liquid_water_ice.csv, which can be found in the EMIT VITALS GitHub repository data folder. We can access and download the k_liquid_water_ice.csv file with the following steps:

1. Follow [this link](https://github.com/nasa/VITALS/blob/main/data/k_liquid_water_ice.csv) to access the k_liquid_water_ice.csv file in the EMIT VITALS repository.
2. Manually download the raw file to your computer.
3. Move the k_liquid_water_ice.csv file into the data directory we made above (`data_dir = r"../data"`). **It is important that the k_liquid_water_ice.csv file is in the data_dir; the calc_ewt and calc_ewt_neon functions expect the CSV file to be in the data_dir.**
4. Run the code in the cell below to open and look at the k_liquid_water_ice.csv file in this notebook. 

The existing [EMIT VITALS CWC tutorial notebook](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#setup) has some helpful details about what this k_liquid_water_ice.csv file is:
>We need some lab measurements of the complex refractive index of liquid water to obtain the wavelength-dependent absorption coefficients. They are calculated by taking four times the product of Pi and the imaginary part of the refractive index, divided by wavelength. The refractive index of liquid water per wavelength is provided by the k_liquid_water_ice.csv in the data folder.

In [ ]:
# Define file path to k_liquid_water_ice.csv file
wp_fp = ("../data/k_liquid_water_ice.csv")

# Read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

# Check k_wi DataFrame
k_wi.head()

### 2. Open NEON and EMIT Reflectance Data

In Tutorial Notebook 01, we downloaded and processed EMIT L2A Reflectance data and NEON Level 3 Spectrometer orthorectified surface bidirectional reflectance - mosaic data. Specifically, we downloaded EMIT reflectance data for one granule from July 31, 2023 and cropped it to the burned and unburned tiles of interest. We downloaded NEON bidirectional reflectance from 2024 for the burned and unburned tiles. We processed the NEON reflectance data in the following ways:

1. scaling the reflectance data by the scale factor (NEON data are saved in an integer format, scaled by 10000, in order to save on space)
2. setting the water vapor absorption windows (defined as "bad band windows") to NaN. Similar to the EMIT data, "good_wavelengths" are provided as one of the Coordinates in the neon_burn_refl_ds xarray dataset, so we can use that information to keep only the valid wavelengths
3. writing the CRS (coordinate reference system information)

We also exported the cropped and processed EMIT and NEON reflectance data to NetCDF files. These NetCDF files were saved in the `../data/refl` directory.

In the code cells below, we are going to define file paths to the cropped and processed EMIT and NEON reflectance. We'll need the file path variables (`neon_burn_fp`, `emit_burn_fp`, etc) for the CWC calculation functions below.

*Note: to define the file path for and open the EMIT and NEON reflectance data, we are using the same code cell 4 times below. Normally, this is repetitive and we would create a function or loop to do this more efficiently. However, since we are just loading in and looking at the datasets and it is not the main focus of this tutorial, we will not focus on making it more efficient.*

In [ ]:
# Check to see that the NetCDF files are in the ../data/refl directory
data_download_tracker(
    # Path to the data_dir
    absolute_soap_path = r'../data/',
    # Folder name in the data_dir that we want to explore
    folder_names = ['refl'],
    # File type we're looking for
    extension = '.nc'
)

In [ ]:
# Define file path to the burned NEON reflectance NetCDF file
# neon_burn_refl_fp = ("../data/refl/neon_burn_refl_float32.nc")
neon_burn_refl_fp = "../data/refl/neon_burn_refl.nc"

# Open NetCDF NEON burned dataset
neon_burn_refl_ds = xr.open_dataset(neon_burn_refl_fp, decode_coords="all")

# Optionally, uncomment to view neon_burn_refl_ds
neon_burn_refl_ds

In [ ]:
# Define file paths to the unburned NEON reflectance NetCDF file
#neon_unburn_refl_fp = ("../data/refl/neon_unburn_refl.nc")
neon_unburn_refl_fp = "../data/refl/neon_unburn_refl.nc"

# Open NetCDF NEON unburned dataset
neon_unburn_refl_ds = xr.open_dataset(neon_unburn_refl_fp, decode_coords="all")

# Optionally, uncomment to view neon_unburn_refl_ds
neon_unburn_refl_ds

In [ ]:
# Define file path to the cropped burned EMIT reflectance NetCDF file

# The file path below is the name of my emit burned reflectance file from my 07 notebook
# emit_burn_refl_fp = ("../data/refl/emit_burn_refl.nc")

# The file path below is the name of the emit burned reflectance file originally saved from the 01 tutorial notebook - doesn't work w/ calc_open_cwc fxn
# emit_burn_refl_fp = "../data/refl/EMIT_L2A_RFL_20230731_SOAP_burned.nc"

# Change name of .nc file in the emit_burn_refl_fp from ../data/refl/EMIT_L2A_RFL_20230731_SOAP_burned.nc to:
emit_burn_refl_fp = "../data/refl/emit_soap_burned.nc"

# Open NetCDF EMIT burned dataset
emit_burn_refl_ds = xr.open_dataset(emit_burn_refl_fp, decode_coords="all")

# Optionally, uncomment to view emit_burn_refl_ds
emit_burn_refl_ds

In [ ]:
# Define file path to the cropped unburned EMIT reflectance NetCDF file

# The file path below is the generic name I input as pseudocode when creating this notebook before 01 tutorial notebook was ready
# emit_unburn_refl_fp = ("../data/refl/emit_unburn_refl.nc")

# The file path below is the name of the emit unburned reflectance file originally saved from the 01 tutorial notebook - doesn't work w/ calc_open_cwc fxn
# emit_unburn_refl_fp = "../data/refl/EMIT_L2A_RFL_20230731_SOAP_unburned.nc"

# Change name of .nc file in the emit_unburn_refl_fp from ../data/refl/EMIT_L2A_RFL_20230731_SOAP_unburned.nc to:
emit_unburn_refl_fp = "../data/refl/emit_soap_unburned.nc"

# Open NetCDF EMIT unburned dataset
emit_unburn_refl_ds = xr.open_dataset(emit_unburn_refl_fp, decode_coords="all")

# Optionally, uncomment to view emit_unburn_refl_ds
emit_unburn_refl_ds

### 3. Calculate Canopy Water Content (CWC)

#### Calculate CWC using the calc_ewt function imported in the beginning

In [ ]:
# Learn about calc_ewt function
help(calc_ewt)

In [ ]:
# Learn about neon_calc_ewt function
help(calc_ewt_neon)

In [ ]:
# Set output directory where results of CWC function will be stored
out_dir = r"../data/cwc/"

**07/28/2025 update** The CWC conditional I have above for EMIT and NEON burn works on it's own. However, having to use it 4 times in a row is very repetitive. It'd be nice if I had a function or a loop to calculate CWC for all 4 scenarious with less repetitive code. Either a function that can be defined once and used 4 times or a loop that goes through all 4 scenarious at once.

**Here is possible work on a function to calculate CWC:**

In [ ]:
def calc_open_cwc(
    dsource_burntype_refl_fp,
    dsource,
    burntype
    # should maybe and calc_ewt parameters here too
):
    dsource_burntype_refl_fp = dsource_burntype_refl_fp
    dsource_burntype_cwc_fp = f"../data/cwc/{dsource}_{burntype}_cwc.nc"
    print(dsource)
    print(burntype)
    print(f'this is the dsource_burntype_refl_fp: {dsource_burntype_refl_fp}')
    # print(f'this is the dsource_burntype_cwc_fp: {dsource_burntype_cwc_fp}')
    # If CWC has already been calculated and saved to a NetCDF file,
    if os.path.exists(dsource_burntype_cwc_fp):
        print('CWC path exists, displaying CWC dataset...')
        # Open the CWC netcdf file
        dsource_burntype_cwc_ds = xr.open_dataset(
            dsource_burntype_cwc_fp, decode_coords="all")
        # Display the CWC dataset
        display(dsource_burntype_cwc_ds)
        print(f'Here is where the CWC dataset NetCDF file is saved: '
              f'{dsource_burntype_cwc_fp}')
    else:
        if 'emit' in dsource_burntype_refl_fp:
            print(f'{dsource_burntype_cwc_fp} does not exist,'
                  f' calculating CWC for {dsource_burntype_refl_fp}...')
            dsource_burntype_cwc_ds = calc_ewt(
                # File path to reflectance dataset
                dsource_burntype_refl_fp,
                out_dir,
                ewt_detection_limit=1.5,
                return_cwc=True
            )
            print(f'CWC for {dsource_burntype_refl_fp} calculated,'
                  f' here is the CWC dataset:')
            display(dsource_burntype_cwc_ds)
            print('\n')
            # COMMENTING OUT NETCDF EXPORT BELOW WHILE STILL IN TESTING/CHECKING PHASE
            # print('Now exporting CWC dataset to a NetCDF file...')
            # dsource_burntype_cwc_ds.to_netcdf(
            #     f"../data/cwc/{dsource}_{burntype}_cwc.nc")
            # print(f'NetCDF file created here:'
            #       f' ../data/cwc/{dsource}_{burntype}_cwc.nc')
        if 'neon' in dsource_burntype_refl_fp:
            print(f'{dsource_burntype_cwc_fp} does not exist,'
                  f' calculating CWC for {dsource_burntype_refl_fp}...')
            dsource_burntype_cwc_ds = calc_ewt_neon(
                # Filepath to reflectance dataset
                dsource_burntype_refl_fp,
                out_dir,
                ewt_detection_limit=1.5,
                return_cwc=True
            )
            print(f'CWC for {dsource_burntype_refl_fp} calculated,'
                  f' here is the CWC dataset:')
            display(dsource_burntype_cwc_ds)
            print('\n')
            print('Now exporting CWC dataset to a NetCDF file...')
            dsource_burntype_cwc_ds.to_netcdf(
                f"../data/cwc/{dsource}_{burntype}_cwc.nc")
            print(f'NetCDF file created here:'
                  f' ../data/cwc/{dsource}_{burntype}_cwc.nc')
            
    return dsource_burntype_cwc_ds

In [ ]:
%%time
neon_burn_cwc_ds = calc_open_cwc(
    neon_burn_refl_fp,
    dsource = r"neon",
    burntype = r"burn"
)

In [ ]:
%%time
neon_unburn_cwc_ds = calc_open_cwc(
    neon_unburn_refl_fp,
    dsource = r"neon",
    burntype = r"unburn"
)

In [ ]:
%%time
emit_burn_cwc_ds = calc_open_cwc(
    emit_burn_refl_fp,
    dsource = r"emit",
    burntype = r"burn"
)

In [ ]:
%%time
emit_unburn_cwc_ds = calc_open_cwc(
    emit_unburn_refl_fp,
    dsource = r"emit",
    burntype = r"unburn"
)

### 3.1 Visualize CWC Datasets - DO THIS NEXT!! also add a section for possible next steps

#### 3.1.1 Interactive plots

In [ ]:
hv.Layout(
    surfrfl_hvplot_image(
        emit_unburn_cwc_ds,
        plottitle=f"EMIT Unburned CWC ({emit_burn_cwc_ds.cwc.units}) DATE HERE",
        cmap='jet_r',
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='longitude', y='latitude'
    )
    +
    surfrfl_hvplot_image(
        neon_unburn_cwc_ds,
        plottitle=f"NEON Unburned CWC ({neon_burn_cwc_ds.cwc.units}) DATE HERE",
        cmap='jet_r',
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='x', y='y'
    )    
    +
    surfrfl_hvplot_image(
        emit_burn_cwc_ds,
        plottitle=f"EMIT Burned CWC ({emit_burn_cwc_ds.cwc.units}) DATE HERE",
        cmap='jet_r',
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='longitude', y='latitude'
    )
    +
    surfrfl_hvplot_image(
        neon_burn_cwc_ds,
        plottitle=f"NEON Burned CWC ({neon_burn_cwc_ds.cwc.units}) DATE HERE",
        cmap='jet_r',
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='x', y='y'
    )
).cols(2)


In [ ]:
# Try plotting neon_burn_cwc_ds using plotting code from Bridget's 06 notebook
neon_burn_cwc_ds.hvplot.image(x='x', y='y',
                              frame_height=405, frame_width=720,
                              fontscale=2,
                              cmap='jet_r',
                              clim = (0,0.5),
                              tiles='ESRI',
                              xlabel='Longitude',
                              ylabel='Latitude',
                              title='NEON CWC - SOAP Burned Tile',
                              crs='EPSG:4326')

#### 3.1.2 Histograms

In [ ]:
# neon_burned_cwc = neon_burn_cwc_ds['cwc'].data.flatten()
# neon_unburned_cwc = neon_unburn_cwc_ds['cwc'].data.flatten()

# plt.figure(figsize=(8, 6))
# plt.hist(neon_burned_cwc, bins=100, alpha=0.5,
#          label='NEON Burned CWC', color='orange')
# plt.hist(neon_unburned_cwc, bins=100, alpha=0.5,
#          label='NEON Unburned CWC', color='green')
# plt.xlim(0, 0.75)
# plt.xlabel('Value')
# plt.ylabel('Frequency')
# plt.title('Histogram of CWC from Burned and Unburned NEON Tiles at SOAP')
# plt.legend()
# plt.show()

In [ ]:
# emit_burned_cwc = neon_burn_cwc_ds['cwc'].data.flatten()
# emit_unburned_cwc = neon_unburn_cwc_ds['cwc'].data.flatten()

# plt.figure(figsize=(8, 6))
# plt.hist(emit_burned_cwc, bins=100, alpha=0.5,
#          label='EMIT Burned CWC', color='orange')
# plt.hist(emit_unburned_cwc, bins=100, alpha=0.5,
#          label='EMIT Unburned CWC', color='green')
# plt.xlim(0, 0.75)
# plt.xlabel('Value')
# plt.ylabel('Frequency')
# plt.title('Histogram of CWC from Burned and Unburned EMIT Tiles at SOAP')
# plt.legend()
# plt.show()

### 4. Compare CWC Datasets

Start w/ histogram comparisons of values - see bridget's 07 notebook. Bridget also created som eKDE (Kernal density plots)

Then think about rescaling and then finding the difference

